## Questão 5 - Dimensão de calendário

### Cenário

O Sr. Almir quer saber: "Qual é o dia da semana (Segunda, Terça...), nas lojas fisicas, temos a pior média de vendas?" para decidir se vale a pena fechar a loja nesses dias.
Um estagiário fez um GROUP BY dia_semana direto na tabela de vendas e disse que o Domingo era ótimo, com média de R$5.000,00. 
O problema: O estagiário esqueceu que em muitos Domingos a loja abriu mas vendeu zero. Como esses dias não existem na tabela de vendas (orders), eles foram ignorados no cálculo da média, inflando o resultado. Precisamos corrigir isso utilizando um calendário de datas (dimensão de datas)

### Premissas obrigatórias:
- O período de análise deve considerar todas as datas entre a menor e a data atual da venda presentes no arquivo.
- A loja esteve aberta em todos os dias do período (inclusive fins de semana).
- Considere apenas as lojas fisicas (= pos)
- Dias sem registro na tabela de vendas devem ser considerados como valor da venda = 0.
- “Vendas diárias” correspondem à soma de valor da venda por dia.
- A média de vendas por dia da semana deve considerar todos os dias do calendário, inclusive os dias sem venda.
- O nome do dia da semana deve ser apresentado em português (Segunda-feira, Terça-feira, etc.).

### Tarefa:
1. Construa uma dimensão de datas utilizando sql


Esta análise foi construída para evitar um viés comum no cálculo de médias diárias: dias em que a loja esteve aberta, mas não realizou vendas, não possuem registros na tabela `orders`.

Caso a tabela `orders` fosse agrupada diretamente pelo dia da semana, seriam considerados apenas os dias com pedidos. Como consequência, a média seria inflada e poderia indicar um desempenho melhor do que o realmente observado.

Para corrigir esse efeito, foi criado um calendário com todas as datas entre a menor e a maior data disponível na base. Em seguida, as vendas físicas diárias foram associadas a esse calendário.

### Fórmulas e funções utilizadas

**Período de análise**

`MIN(created_at::date)` identifica a primeira data disponível na tabela de pedidos.

`MAX(created_at::date)` identifica a última data disponível na tabela de pedidos.

Essas funções definem os limites do calendário analisado.

**Calendário completo**

`generate_series(data_inicial, data_final, INTERVAL '1 day')` gera uma linha para cada dia do período, incluindo fins de semana e dias sem pedidos.

**Vendas diárias das lojas físicas**

`SUM(total)` calcula o faturamento diário das vendas realizadas pelo canal físico, filtrado por `channel = 'pos'`.

**Dias sem vendas**

`COALESCE(vendas_diarias, 0)` substitui valores nulos por zero quando não existe venda registrada para uma data do calendário.

**Média por dia da semana**

`AVG(vendas_diarias)` calcula a média das vendas diárias para cada dia da semana, considerando também os dias cujo faturamento foi zero.

A função `EXTRACT(ISODOW FROM data)` identifica o número do dia da semana de 1 a 7, em que 1 representa segunda-feira e 7 representa domingo. Esse número foi utilizado para garantir a ordenação cronológica correta.


```sql
-- ============================================================
-- QUESTÃO 5 - Média de vendas das lojas físicas por dia da semana
-- Objetivo: incluir no cálculo os dias em que não houve venda.
-- ============================================================

WITH periodo_analise AS (
    -- Identifica a primeira e a última data disponíveis na base.
    -- O recorte será utilizado para criar o calendário completo.
    SELECT
        MIN(created_at::date) AS data_inicial,
        MAX(created_at::date) AS data_final
    FROM orders
),

calendario AS (
    -- Gera uma linha para cada data entre o início e o fim do período.
    -- Dessa forma, também são incluídos fins de semana e dias sem pedidos.
    SELECT
        serie.data::date AS data,

        -- ISODOW retorna 1 para segunda-feira e 7 para domingo.
        EXTRACT(ISODOW FROM serie.data)::int AS numero_dia_semana,

        -- Converte o número do dia no nome em português.
        CASE EXTRACT(ISODOW FROM serie.data)::int
            WHEN 1 THEN 'Segunda-feira'
            WHEN 2 THEN 'Terça-feira'
            WHEN 3 THEN 'Quarta-feira'
            WHEN 4 THEN 'Quinta-feira'
            WHEN 5 THEN 'Sexta-feira'
            WHEN 6 THEN 'Sábado'
            WHEN 7 THEN 'Domingo'
        END AS dia_semana
    FROM periodo_analise

    -- generate_series cria uma sequência diária entre as datas definidas.
    CROSS JOIN LATERAL generate_series(
        data_inicial,
        data_final,
        INTERVAL '1 day'
    ) AS serie(data)
),

vendas_diarias_pos AS (
    -- Filtra apenas vendas do canal físico e soma o faturamento por dia.
    SELECT
        created_at::date AS data,
        SUM(total) AS vendas_diarias
    FROM orders
    WHERE channel = 'pos'
    GROUP BY created_at::date
),

vendas_com_calendario AS (
    -- Mantém todas as datas do calendário, mesmo quando não há venda.
    SELECT
        c.data,
        c.numero_dia_semana,
        c.dia_semana,

        -- Quando não há venda para a data, o valor nulo é substituído por zero.
        COALESCE(v.vendas_diarias, 0) AS vendas_diarias
    FROM calendario AS c
    LEFT JOIN vendas_diarias_pos AS v
        ON c.data = v.data
)

-- Calcula a média de vendas diárias para cada dia da semana.
SELECT
    dia_semana,
    ROUND(AVG(vendas_diarias), 2) AS media_vendas_diarias
FROM vendas_com_calendario
GROUP BY
    numero_dia_semana,
    dia_semana

-- Ordena da menor para a maior média, facilitando identificar o pior dia.
ORDER BY
    media_vendas_diarias ASC,
    numero_dia_semana ASC;

## Explique
### Por que é necessário utilizar uma tabela de datas (calendário) em vez de agrupar diretamente a tabela de vendas? 

É necessário utilizar uma tabela de datas porque a tabela `orders` registra somente os dias em que houve pelo menos um pedido. Quando um agrupamento é feito diretamente nessa tabela, dias sem vendas não aparecem no resultado e são excluídos do cálculo da média.

Isso gera viés na análise. Por exemplo, se a loja ficou aberta em vários domingos, mas realizou vendas em apenas alguns deles, um `GROUP BY` direto consideraria somente os domingos com pedidos. Como os domingos sem venda seriam ignorados, a média calculada ficaria artificialmente elevada.

A dimensão de calendário contém todos os dias do período analisado, inclusive fins de semana e datas sem movimentação. Com o `LEFT JOIN` entre o calendário e as vendas diárias, os dias sem pedido são preservados e recebem valor igual a zero por meio de `COALESCE`.

Dessa forma, a média de vendas por dia da semana passa a refletir todos os dias em que a loja esteve aberta, representando de forma mais confiável a demanda real da operação.


### O que aconteceria com a média de vendas se um dia da semana tivesse muitos dias sem nenhuma venda registrada?

Se um dia da semana apresentar muitos dias sem nenhuma venda registrada, sua média de vendas será reduzida, pois esses dias devem ser incluídos no cálculo com valor igual a zero.

Por exemplo, se ocorrerem vendas de R$ 5.000,00 em dois domingos e existirem outros dois domingos sem vendas, a média será:

`(5.000 + 5.000 + 0 + 0) / 4 = 2.500`

Portanto, quanto maior for a quantidade de dias sem venda, menor será a média diária daquele dia da semana. Essa inclusão é necessária para representar a operação real da loja, já que foi considerado que ela permaneceu aberta em todos os dias do período.